# 01_translate_svg

Second step in the TBE SVG translation workflow.

This notebook loads the translation units created by `00_open_and_inspect_svg.ipynb`, sends those units to an LLM in packetized JSON requests, validates the returned translations, saves translated JSON output, and writes translated SVG files.

The overall pattern follows the earlier BST (*Bible Structure and Timeline*) SVG workflow. TBE uses the same two-notebook structure, but the source SVGs contain more complex fragmented `<tspan>` text. Because of that, this notebook also includes an optional reconstruction path that collapses fragmented tspans into a single translated text span when writing translated SVG output.

Major steps:

- configure source and target languages, model, API client, and optional domain context
- load `json_files/translation_units.json`
- packetize translation units for LLM requests
- send and validate one test packet
- run all packets and save timestamped translated JSON
- apply translations back into SVG files
- export CSV, XLSX, and Markdown review tables
- optionally write collapsed-tspan SVG outputs to `svg_output_files_collapsed_tspans/`

In [1]:
# Set languages and LLM model

source_language = 'English' # for LLM prompting
target_language = 'Simplified Chinese' # for LLM prompting

gemini_model_name = 'gemini-3.1-pro-preview' # Primary translator


In [ ]:
# Load API key
# API key must be in a .env file in working directory

import os
from dotenv import load_dotenv
from google import genai

# Load .env once
load_dotenv()

# Force Gemini key usage (paid access)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not found in environment")

# OPTIONAL: prevent accidental fallback
# This avoids the SDK silently choosing GOOGLE_API_KEY
os.environ.pop("GOOGLE_API_KEY", None)

# Create a single reusable client
genai_client = genai.Client(api_key=GEMINI_API_KEY)

# print(f"Gemini client initialized with key: {GEMINI_API_KEY[:4]}…{GEMINI_API_KEY[-4:]}")


## System message
**CAUTION:** Do not customize the following cell

In [3]:
# do not customize this cell

def build_svg_units_system_instruction(
    source_language: str,
    target_language: str,
    domain_context: str | None = None,
) -> str:
    custom_block = ""
    if domain_context:
        custom_block = f"""
   - Domain context:
     {domain_context}
""".rstrip()

    return f"""
You are a professional translator.

You will receive ONE JSON object with this structure:

{{
  "units": [
    {{
      "unit_key": "<string>",
      "group_stack": "<string describing the SVG layer/group context>",
      "source_text": "<text in {source_language}>"
    }},
    ...
  ]
}}

Translate each `source_text` from {source_language} into {target_language}.
The `group_stack` is context to help you choose appropriate wording, capitalization, and abbreviations.

You must respond with ONE valid JSON object of the form:

{{
  "units": [
    {{
      "unit_key": "<same unit_key as input>",
      "translated_text": "<translated text in {target_language}>"
    }},
    ...
  ]
}}

REQUIREMENTS

1) JSON contract:
   - Return exactly one top-level JSON object with exactly one key: "units".
   - "units" must have the same number of entries as the input, in the same order.
   - Each output entry must contain exactly two keys: "unit_key" and "translated_text".
   - Do not include "group_stack" or "source_text" in the output.
   - Output must be valid JSON: no trailing commas, no comments, no code fences, no extra text.
   - Each "unit_key" must appear exactly once in the output. Do not duplicate, omit, or invent unit_key values.
   - The i-th output unit must correspond to the i-th input unit (same order).

2) Text handling:
   - Preserve numbers, punctuation, and abbreviations when meaningful.
   - Keep proper nouns unchanged unless a standard {target_language} form exists.
   - If `source_text` is already primarily in {target_language}, copy it unchanged.
   - Use wording and idiom appropriate to the source context indicated by `group_stack`.{custom_block}

3) Numbers and mixed text:
   - Preserve all numbers exactly as they appear in the source_text.
   - When source_text contains both words and numbers, translate only the words and keep all numbers unchanged and in the same relative position.
   - Do not convert numbers to words, do not reformat them, and do not add or remove numeric content.

4) Brevity:
   - These are short labels for a graphic. Prefer concise translations that fit similar space.

Your entire response must be ONLY the JSON object described above.
""".strip()

### Domain context (optional)
- Update `domain_context` below if this SVG set needs brief project-specific translation guidance.
- Keep it short and focused on terminology or interpretive context specific to this project.

In [4]:
primary_system_message = build_svg_units_system_instruction(
    source_language=source_language,
    target_language=target_language,
    domain_context=(
        "This material contains Biblical and historical labels. "
        "Use terminology natural to a Protestant and Evangelical Christian context. "
        "Preserve established Biblical names and standard historical/Biblical terms in the target language."
    ),
)

In [5]:
print(primary_system_message)

You are a professional translator.

You will receive ONE JSON object with this structure:

{
  "units": [
    {
      "unit_key": "<string>",
      "group_stack": "<string describing the SVG layer/group context>",
      "source_text": "<text in English>"
    },
    ...
  ]
}

Translate each `source_text` from English into Simplified Chinese.
The `group_stack` is context to help you choose appropriate wording, capitalization, and abbreviations.

You must respond with ONE valid JSON object of the form:

{
  "units": [
    {
      "unit_key": "<same unit_key as input>",
      "translated_text": "<translated text in Simplified Chinese>"
    },
    ...
  ]
}

REQUIREMENTS

1) JSON contract:
   - Return exactly one top-level JSON object with exactly one key: "units".
   - "units" must have the same number of entries as the input, in the same order.
   - Each output entry must contain exactly two keys: "unit_key" and "translated_text".
   - Do not include "group_stack" or "source_text" in the

In [6]:
print(repr(primary_system_message))

'You are a professional translator.\n\nYou will receive ONE JSON object with this structure:\n\n{\n  "units": [\n    {\n      "unit_key": "<string>",\n      "group_stack": "<string describing the SVG layer/group context>",\n      "source_text": "<text in English>"\n    },\n    ...\n  ]\n}\n\nTranslate each `source_text` from English into Simplified Chinese.\nThe `group_stack` is context to help you choose appropriate wording, capitalization, and abbreviations.\n\nYou must respond with ONE valid JSON object of the form:\n\n{\n  "units": [\n    {\n      "unit_key": "<same unit_key as input>",\n      "translated_text": "<translated text in Simplified Chinese>"\n    },\n    ...\n  ]\n}\n\nREQUIREMENTS\n\n1) JSON contract:\n   - Return exactly one top-level JSON object with exactly one key: "units".\n   - "units" must have the same number of entries as the input, in the same order.\n   - Each output entry must contain exactly two keys: "unit_key" and "translated_text".\n   - Do not include 

In [7]:
print("Domain context present:", "Protestant and Evangelical Christian context" in primary_system_message)

Domain context present: True


In [8]:
print("Using system message with length:", len(primary_system_message))

Using system message with length: 2418


In [9]:
def gemini_generate(payload_json_str: str, timeout: int = 120):
    return genai_client.models.generate_content(
        model=gemini_model_name,
        contents=payload_json_str,
        system_instruction=primary_system_message,
        request_options={"timeout": timeout},
    )

print("Gemini client ready:", genai_client)


Gemini client ready: <google.genai.client.Client object at 0x0000011B9D989A10>


## Import JSON payload

In [10]:
# Set directories

from pathlib import Path

PROJECT_ROOT = Path.cwd()
SVG_SOURCE_DIR = PROJECT_ROOT / "svg_source_files"
SVG_OUTPUT_DIR = PROJECT_ROOT / "svg_output_files"
JSON_DIR = PROJECT_ROOT / "json_files"

assert SVG_SOURCE_DIR.exists(), f"Missing folder: {SVG_SOURCE_DIR}"
assert SVG_OUTPUT_DIR.exists(), f"Missing folder: {SVG_OUTPUT_DIR}"
assert JSON_DIR.exists(), f"Missing folder: {JSON_DIR}"

print("PROJECT_ROOT:", PROJECT_ROOT.name)
print("SVG_SOURCE_DIR:", SVG_SOURCE_DIR.relative_to(PROJECT_ROOT.parent))
print("JSON_DIR:", JSON_DIR.relative_to(PROJECT_ROOT.parent))
print("SVG_OUTPUT_DIR:", SVG_OUTPUT_DIR.relative_to(PROJECT_ROOT.parent))

PROJECT_ROOT: tbe
SVG_SOURCE_DIR: tbe\svg_source_files
JSON_DIR: tbe\json_files
SVG_OUTPUT_DIR: tbe\svg_output_files


In [11]:
import json
from pathlib import Path
import pandas as pd

# --- locate and load the translation-units JSON produced in the previous notebook ---

units_path = JSON_DIR / "translation_units.json"
assert units_path.exists(), f"Missing file: {units_path}"

units = json.loads(units_path.read_text(encoding="utf-8"))
assert isinstance(units, list), "Expected a JSON array of unit records."

df_units = pd.DataFrame(units)
required_cols = {"unit_key", "unit_type", "source_file", "group_stack", "element_path", "source_text"}
missing = required_cols - set(df_units.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

df_units["source_text"] = df_units["source_text"].fillna("").astype(str)
df_units = df_units[df_units["source_text"].str.strip().ne("")].copy()
df_units.reset_index(drop=True, inplace=True)

print("Loaded units:", len(df_units))
print("Files:", df_units["source_file"].nunique())
df_units.head(10)


Loaded units: 341
Files: 4


,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,skip_reason
0,42415ca24611ff65186249ab11a15d368b1530d6,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Adam 4174,
1,63b52a9dd6f89c753d7d5daf49710893a72629c7,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Enoch 3552,
2,c7cbff308d158d2ed4d94d7294e625fc2a2319dc,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Enosh 3939,
3,872ee56f45524087e305b6a4cdcd8d5a1fce1159,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Jared 3714,
4,e40bcebd3c831a8e80a629f41bf3e7104d2b251c,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Kenan 3849,
5,3c23d44eb081747c7ba965be21bee62ddd2676b5,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Lamech 3300,
6,2c5b67b2cdc4c79daa161d826445b83c7b614c89,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Mahalalel 3779,
7,7ee468a69144fb5233179d42fe06d7a69bdb3f57,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Methuselah 3487,
8,1b1910bf74e9a176beec1ccbd18517c88b4db05c,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Noah 3118,
9,1c1a27384f36d007b684cd5c61c9396394f7a395,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Seth 4044,


## Prepare units
### Packet sizing
- These settings control how translation units are grouped into API requests.
- In most cases, leave these values unchanged.  
Only reduce them if requests are failing, timing out, or returning malformed/incomplete JSON.  
Increase them only if you are intentionally testing larger batches and understand the risk of oversized requests or harder-to-debug failures.
- A small final packet is normal and expected with this packing method.

In [12]:
# --- packetize units into request-sized chunks (simple char-based packing) ---
# Adjust these if you want larger/smaller packets; char-based is predictable across models.

# MAX_CHARS_PER_PACKET = 12000   # conservative for prompts + JSON overhead
# MAX_UNITS_PER_PACKET = 200     # hard cap to keep responses manageable

MAX_CHARS_PER_PACKET = 6000
MAX_UNITS_PER_PACKET = 100

def packetize_units(df: pd.DataFrame,
                    max_chars: int = MAX_CHARS_PER_PACKET,
                    max_units: int = MAX_UNITS_PER_PACKET):
    packets = []
    current = []
    current_chars = 0

    # Stable ordering helps reproducibility and diffing
    df_sorted = df.sort_values(["source_file", "group_stack", "element_path", "unit_type", "tspan_idx"], na_position="last")

    for _, r in df_sorted.iterrows():
        rec = {
            "unit_key": r["unit_key"],
            "group_stack": r["group_stack"],
            "source_text": r["source_text"],
        }
        # Estimate size as JSON string length
        rec_chars = len(json.dumps(rec, ensure_ascii=False))
        if rec_chars > max_chars:
            raise ValueError(f"Single unit exceeds max_chars ({rec_chars} > {max_chars}): {rec['unit_key']}")

        # Start new packet if needed
        if current and ((current_chars + rec_chars) > max_chars or (len(current) >= max_units)):
            packets.append(current)
            current = []
            current_chars = 0

        current.append(rec)
        current_chars += rec_chars

    if current:
        packets.append(current)

    return packets

packets = packetize_units(df_units)

print("Packets:", len(packets))
print("Units per packet (first 10):", [len(p) for p in packets[:10]])
print("Approx chars per packet (first 3):", [sum(len(json.dumps(u, ensure_ascii=False)) for u in p) for p in packets[:3]])


Packets: 12
Units per packet (first 10): [34, 30, 29, 35, 21, 27, 33, 36, 22, 19]
Approx chars per packet (first 3): [5931, 5807, 5828]


In [13]:
# --- build request payloads ready to send to Gemini (each payload is a single JSON object) ---
request_payloads = [{"units": p} for p in packets]

# quick peek at one payload
print("Example payload keys:", request_payloads[0].keys())
print("Example payload unit count:", len(request_payloads[0]["units"]))
print(json.dumps(request_payloads[0]["units"][0], ensure_ascii=False, indent=2))


Example payload keys: dict_keys(['units'])
Example payload unit count: 34
{
  "unit_key": "42415ca24611ff65186249ab11a15d368b1530d6",
  "group_stack": "AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Adam",
  "source_text": "Adam 4174"
}


In [14]:
# inspect an entry

request_payloads[4]["units"][1]

{'unit_key': 'b8da2b77bf0a86a2002e371c9351763357ab7182',
 'group_stack': 'AB3 / ab3_x5F_biblical / ab3_x5F_biblical_x5F_text / ab3_x5F_biblical_x5F_text_x5F_booknames / ab3_x5F_biblical_x5F_text_x5F_booknames_x5F_edom',
 'source_text': 'Obadiah'}

### Inspect payload
- Pick a packet with a small number of entries using `packet_index`

In [15]:
# inspect payload
# pick one with a small number of entries

packet_index = 4

request_payload = {"units": packets[packet_index]}

request_debug = {
    "model": gemini_model_name,
    "system_instruction": primary_system_message,
    "contents": request_payload,   # dict, not string
}

from pprint import pprint
pprint(request_debug)

{'contents': {'units': [{'group_stack': 'AB3 / ab3_x5F_biblical / '
                                        'ab3_x5F_biblical_x5F_text / '
                                        'ab3_x5F_biblical_x5F_text_x5F_booknames',
                         'source_text': 'Lev-Dt',
                         'unit_key': '2841603a54a8cd3a3f0d3165acddfb242c17dded'},
                        {'group_stack': 'AB3 / ab3_x5F_biblical / '
                                        'ab3_x5F_biblical_x5F_text / '
                                        'ab3_x5F_biblical_x5F_text_x5F_booknames '
                                        '/ '
                                        'ab3_x5F_biblical_x5F_text_x5F_booknames_x5F_edom',
                         'source_text': 'Obadiah',
                         'unit_key': 'b8da2b77bf0a86a2002e371c9351763357ab7182'},
                        {'group_stack': 'AB3 / ab3_x5F_biblical / '
                                        'ab3_x5F_biblical_x5F_text / '
               

### Send one test packet

In [16]:
import json
import time
from google.genai import types

print("Preparing request...")
print("Model:", gemini_model_name)
print("Units in payload:", len(request_payload["units"]))
print("Sending request to Gemini...")

t0 = time.time()

response = genai_client.models.generate_content(
    model=gemini_model_name,
    contents=json.dumps(request_payload, ensure_ascii=False),
    config=types.GenerateContentConfig(
        system_instruction=primary_system_message,
    ),
)

elapsed = time.time() - t0
print(f"Response received in {elapsed:.2f} seconds")

print("Preview of response text:")
print(response.text[:2000])

Preparing request...
Model: gemini-3.1-pro-preview
Units in payload: 21
Sending request to Gemini...
Response received in 32.06 seconds
Preview of response text:
```json
{
  "units": [
    {
      "unit_key": "2841603a54a8cd3a3f0d3165acddfb242c17dded",
      "translated_text": "利未记-申命记"
    },
    {
      "unit_key": "b8da2b77bf0a86a2002e371c9351763357ab7182",
      "translated_text": "俄巴底亚书"
    },
    {
      "unit_key": "f70d26ec6bbeadbab554450725e745739d49ae92",
      "translated_text": "< 向以东发预言的先知"
    },
    {
      "unit_key": "9a80a4639cd7c3d05da71bc032dba7cc835aff21",
      "translated_text": "诗篇"
    },
    {
      "unit_key": "752d3bb1eb832fc895d48ca57dee3a8fc9d4f54e",
      "translated_text": "雅歌"
    },
    {
      "unit_key": "43628cf6a67e27199a78afcf6f4c1b6e7f24549b",
      "translated_text": "箴言"
    },
    {
      "unit_key": "9e261e488c403422d1faf1a01673aac8fd4b15c8",
      "translated_text": "传道书"
    },
    {
      "unit_key": "e0a761bc129afde400916109ff5c097d65978

In [17]:
# strip code fences
# One-off cleanup + parse for Gemini responses wrapped in ```json ... ```
import json

raw = response.text

# Strip leading/trailing whitespace first
s = raw.strip()

# If wrapped in fenced code block, remove the fences
if s.startswith("```"):
    lines = s.splitlines()
    # drop first line: ``` or ```json
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    # drop last line: ```
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    s = "\n".join(lines).strip()

# Now parse JSON
parsed = json.loads(s)

# Basic validation
assert "units" in parsed, "Missing top-level key 'units'"
assert len(parsed["units"]) == len(request_payload["units"]), (
    f"Unit count mismatch: got {len(parsed['units'])}, expected {len(request_payload['units'])}"
)

print("Parsed OK | units:", len(parsed["units"]))
print("First 2:", parsed["units"][:2])

Parsed OK | units: 21
First 2: [{'unit_key': '2841603a54a8cd3a3f0d3165acddfb242c17dded', 'translated_text': '利未记-申命记'}, {'unit_key': 'b8da2b77bf0a86a2002e371c9351763357ab7182', 'translated_text': '俄巴底亚书'}]


In [18]:
# verify packets (input vs model output)
def validate_units(result: dict, input_payload: dict):
    out_units = result.get("units", [])
    in_units = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    assert len(out_keys) == len(in_keys), f"Count mismatch: out={len(out_keys)} in={len(in_keys)}"
    assert out_keys == in_keys, "unit_key sequence mismatch (duplicates, missing, or re-ordered keys)."

# use the variables you actually have right now
validate_units(parsed, request_payload)
print("Validation OK")


Validation OK


### Run all packets
#### Load helper functions

In [19]:
from collections import Counter

def debug_unit_key_mismatch(result: dict, input_payload: dict, n_show: int = 20):
    out_units = result.get("units", [])
    in_units  = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    out_counts = Counter(out_keys)
    in_counts  = Counter(in_keys)

    missing = [k for k in in_counts if out_counts[k] == 0]
    extra   = [k for k in out_counts if in_counts[k] == 0]
    dup_out = [k for k, c in out_counts.items() if c > 1]
    dup_in  = [k for k, c in in_counts.items() if c > 1]

    print("Input units:", len(in_keys), "| Output units:", len(out_keys))
    print("Missing keys (in input but not output):", len(missing))
    print("Extra keys (in output but not input):", len(extra))
    print("Duplicates in output:", len(dup_out))
    print("Duplicates in input:", len(dup_in))

    if missing:
        print("\nFirst missing keys:")
        for k in missing[:n_show]:
            print("  ", k)

    if extra:
        print("\nFirst extra keys:")
        for k in extra[:n_show]:
            print("  ", k)

    if dup_out:
        print("\nFirst duplicated output keys:")
        for k in dup_out[:n_show]:
            print("  ", k, "count=", out_counts[k])

    # show first index where order differs (if lengths match)
    if len(out_keys) == len(in_keys):
        for i, (ok, ik) in enumerate(zip(out_keys, in_keys)):
            if ok != ik:
                print(f"\nFirst order mismatch at index {i}:")
                print("  expected:", ik)
                print("  got     :", ok)
                break


In [20]:
from collections import Counter

def debug_unit_key_mismatch(result: dict, input_payload: dict, n_show: int = 20):
    out_units = result.get("units", [])
    in_units  = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    out_counts = Counter(out_keys)
    in_counts  = Counter(in_keys)

    missing = [k for k in in_counts if out_counts[k] == 0]
    extra   = [k for k in out_counts if in_counts[k] == 0]
    dup_out = [k for k, c in out_counts.items() if c > 1]
    dup_in  = [k for k, c in in_counts.items() if c > 1]

    print("Input units:", len(in_keys), "| Output units:", len(out_keys))
    print("Missing keys (in input but not output):", len(missing))
    print("Extra keys (in output but not input):", len(extra))
    print("Duplicates in output:", len(dup_out))
    print("Duplicates in input:", len(dup_in))

    if missing:
        print("\nFirst missing keys:")
        for k in missing[:n_show]:
            print("  ", k)

    if extra:
        print("\nFirst extra keys:")
        for k in extra[:n_show]:
            print("  ", k)

    if dup_out:
        print("\nFirst duplicated output keys:")
        for k in dup_out[:n_show]:
            print("  ", k, "count=", out_counts[k])

    # show first index where order differs (if lengths match)
    if len(out_keys) == len(in_keys):
        for i, (ok, ik) in enumerate(zip(out_keys, in_keys)):
            if ok != ik:
                print(f"\nFirst order mismatch at index {i}:")
                print("  expected:", ik)
                print("  got     :", ok)
                break


In [21]:
import json
import time
import re
import pandas as pd
import httpx
from google.genai import types, errors as genai_errors

def extract_json_object(text: str) -> dict:
    t = (text or "").strip()
    t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"\s*```$", "", t)
    return json.loads(t)

def generate_with_retry(client, model_name: str, system_prompt: str, payload_obj: dict,
                        retries: int = 4, backoff_s: float = 5.0):
    payload_str = json.dumps(payload_obj, ensure_ascii=False)
    last_err = None

    for attempt in range(1, retries + 1):
        try:
            return client.models.generate_content(
                model=model_name,
                contents=payload_str,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                ),
            )

        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            last_err = e
            status = getattr(e, "status_code", None)
            msg = str(e)
            transient = (
                status in {429, 500, 503, 504}
                or "RESOURCE_EXHAUSTED" in msg
                or "UNAVAILABLE" in msg
                or "DEADLINE" in msg
            )
            if (attempt == retries) or (not transient):
                raise
            print(f"retry {attempt}/{retries} after API error: {type(e).__name__}: {e}")
            time.sleep(backoff_s * attempt)

        except (httpx.RemoteProtocolError, httpx.ReadTimeout, httpx.ConnectTimeout,
                httpx.ConnectError, httpx.NetworkError, httpx.TransportError) as e:
            last_err = e
            if attempt == retries:
                raise
            print(f"retry {attempt}/{retries} after transport error: {type(e).__name__}: {e}")
            time.sleep(backoff_s * attempt)

    raise last_err

In [22]:
from collections import Counter

def validate_units_keys_only(result: dict, input_payload: dict):
    """
    Validates that output contains exactly the same unit_keys as input,
    with the same counts, regardless of ordering.
    """
    out_units = result.get("units", [])
    in_units  = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    assert None not in out_keys, "Output contains a unit missing unit_key."
    assert None not in in_keys,  "Input contains a unit missing unit_key."

    out_counts = Counter(out_keys)
    in_counts  = Counter(in_keys)

    missing = [k for k in in_counts if out_counts[k] == 0]
    extra   = [k for k in out_counts if in_counts[k] == 0]
    wrong_count = [k for k in in_counts if out_counts[k] != in_counts[k]]

    assert not missing, f"Missing unit_keys in output (first 5): {missing[:5]}"
    assert not extra,   f"Extra unit_keys in output (first 5): {extra[:5]}"
    assert not wrong_count, f"Count mismatch for some keys (first 5): {wrong_count[:5]}"

def reorder_units_to_input_order(result: dict, input_payload: dict) -> dict:
    """
    Reorders result["units"] to match the order of input_payload["units"].
    Assumes validate_units_keys_only has passed.
    """
    out_units = result.get("units", [])
    in_units  = input_payload["units"]

    buckets = {}
    for u in out_units:
        k = u["unit_key"]
        buckets.setdefault(k, []).append(u)

    ordered = []
    for u_in in in_units:
        k = u_in["unit_key"]
        ordered.append(buckets[k].pop(0))

    leftovers = sum(len(v) for v in buckets.values())
    assert leftovers == 0, "Internal reorder error: leftover output units exist."

    return {"units": ordered}

In [25]:
# --- run all packets ---
all_out = []

for i, payload in enumerate(request_payloads):
    print(f"Packet {i+1}/{len(request_payloads)} ...", end=" ", flush=True)

    try:
        resp = generate_with_retry(
            client=genai_client,
            model_name=gemini_model_name,
            system_prompt=primary_system_message,
            payload_obj=payload,
            retries=4,
            backoff_s=5,
        )

        result = extract_json_object(resp.text)

        validate_units_keys_only(result, payload)
        result = reorder_units_to_input_order(result, payload)

        all_out.extend(result["units"])

        print("OK")

    except Exception as e:
        print(f"FAILED | {type(e).__name__}: {e}")
        raise

df_trans = pd.DataFrame(all_out)
print("Total translated units:", len(df_trans))
df_trans.head(10)

Packet 1/12 ... OK
Packet 2/12 ... OK
Packet 3/12 ... OK
Packet 4/12 ... OK
Packet 5/12 ... OK
Packet 6/12 ... OK
Packet 7/12 ... OK
Packet 8/12 ... OK
Packet 9/12 ... OK
Packet 10/12 ... OK
Packet 11/12 ... OK
Packet 12/12 ... OK
Total translated units: 341


,unit_key,translated_text
0,42415ca24611ff65186249ab11a15d368b1530d6,亚当 4174
1,63b52a9dd6f89c753d7d5daf49710893a72629c7,以诺 3552
2,c7cbff308d158d2ed4d94d7294e625fc2a2319dc,以挪士 3939
3,872ee56f45524087e305b6a4cdcd8d5a1fce1159,雅列 3714
4,e40bcebd3c831a8e80a629f41bf3e7104d2b251c,该南 3849
5,3c23d44eb081747c7ba965be21bee62ddd2676b5,拉麦 3300
6,2c5b67b2cdc4c79daa161d826445b83c7b614c89,玛勒列 3779
7,7ee468a69144fb5233179d42fe06d7a69bdb3f57,玛土撒拉 3487
8,1b1910bf74e9a176beec1ccbd18517c88b4db05c,挪亚 3118
9,1c1a27384f36d007b684cd5c61c9396394f7a395,塞特 4044


### Save translated phrases to timestamped json file

In [26]:
from datetime import datetime
import re
import json

def slugify_for_filename(s: str) -> str:
    """
    Make a filesystem-friendly slug from an arbitrary label.
    - Converts any non-alphanumeric/underscore runs to a single underscore
    - Trims leading/trailing underscores
    """
    s = (s or "").strip()
    return re.sub(r"[^\w]+", "_", s).strip("_")

def timestamp_yyyymmdd_hhmm(dt: datetime | None = None) -> str:
    dt = dt or datetime.now()
    return dt.strftime("%Y%m%d_%H%M")

ts = timestamp_yyyymmdd_hhmm()
lang_slug = slugify_for_filename(target_language)

translations_out_path = JSON_DIR / f"translations_{lang_slug}_{ts}.json"

print("lang_slug:", lang_slug)
print("timestamp:", ts)

translations_out_path.write_text(
    json.dumps(all_out, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Wrote translations:", translations_out_path.relative_to(PROJECT_ROOT))

lang_slug: Simplified_Chinese
timestamp: 20260516_0940
Wrote translations: json_files\translations_Simplified_Chinese_20260516_0940.json


## Swap out phrases and save to svg

In [27]:
# unit_key → translated_text (FROM MEMORY)
trans_map = {r["unit_key"]: r["translated_text"] for r in all_out}

print("Translations in memory:", len(trans_map))


Translations in memory: 341


In [28]:
import pandas as pd

trans_map = {r["unit_key"]: r["translated_text"] for r in all_out}

df_apply = df_units.copy()
df_apply["translated_text"] = df_apply["unit_key"].map(trans_map)

print("Rows in df_apply:", len(df_apply))
print("Translated rows:", df_apply["translated_text"].notna().sum())
print("Missing translations:", df_apply["translated_text"].isna().sum())

df_apply[["source_file", "unit_key", "unit_type", "element_path", "tspan_idx", "translated_text"]].head(10)

Rows in df_apply: 341
Translated rows: 341
Missing translations: 0


,source_file,unit_key,unit_type,element_path,tspan_idx,translated_text
0,tbe_01.svg,42415ca24611ff65186249ab11a15d368b1530d6,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,亚当 4174
1,tbe_01.svg,63b52a9dd6f89c753d7d5daf49710893a72629c7,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,以诺 3552
2,tbe_01.svg,c7cbff308d158d2ed4d94d7294e625fc2a2319dc,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,以挪士 3939
3,tbe_01.svg,872ee56f45524087e305b6a4cdcd8d5a1fce1159,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,雅列 3714
4,tbe_01.svg,e40bcebd3c831a8e80a629f41bf3e7104d2b251c,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,该南 3849
5,tbe_01.svg,3c23d44eb081747c7ba965be21bee62ddd2676b5,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,拉麦 3300
6,tbe_01.svg,2c5b67b2cdc4c79daa161d826445b83c7b614c89,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,玛勒列 3779
7,tbe_01.svg,7ee468a69144fb5233179d42fe06d7a69bdb3f57,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,玛土撒拉 3487
8,tbe_01.svg,1b1910bf74e9a176beec1ccbd18517c88b4db05c,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,挪亚 3118
9,tbe_01.svg,1c1a27384f36d007b684cd5c61c9396394f7a395,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,塞特 4044


In [29]:
from datetime import datetime
from pathlib import Path
from lxml import etree
import re

SVG_NS = "http://www.w3.org/2000/svg"
NS = {"svg": SVG_NS}

target_language_slug = re.sub(r"[^A-Za-z0-9]+", "_", target_language).strip("_").lower()
timestamp = datetime.now().strftime("%Y%m%d_%H%M")


def localname(tag) -> str:
    if not isinstance(tag, str):
        return ""
    return tag.split("}", 1)[1] if tag.startswith("{") else tag

def find_by_element_path(root, path: str):
    """
    Resolve paths of the form:
    svg/g[1]/text[3]
    svg/g#Layer_1/text#text42
    svg/g[2]/text[1]/tspan[3]
    """
    if not path:
        return None

    parts = path.split("/")
    cur = root

    # first part should describe the root svg
    first = parts[0]
    if not first.startswith("svg"):
        return None

    for part in parts[1:]:
        m = re.fullmatch(r"([A-Za-z0-9_:-]+)(?:#(.+)|\[(\d+)\])?", part)
        if not m:
            return None

        tag, el_id, idx = m.groups()
        candidates = [
            child for child in cur
            if isinstance(getattr(child, "tag", None), str) and localname(child.tag) == tag
        ]

        if el_id is not None:
            match = None
            for child in candidates:
                if child.get("id") == el_id:
                    match = child
                    break
            if match is None:
                return None
            cur = match

        elif idx is not None:
            idx0 = int(idx) - 1  # stored paths are 1-based
            if idx0 < 0 or idx0 >= len(candidates):
                return None
            cur = candidates[idx0]

        else:
            if not candidates:
                return None
            cur = candidates[0]

    return cur


def set_text_preserve_none(el, new_text: str):
    """
    Replace text content safely.
    """
    el.text = "" if new_text is None else str(new_text)


def apply_translations_to_svg(
    svg_path: Path,
    units_for_file: pd.DataFrame,
    target_language_slug: str,
    timestamp: str,
    output_dir: Path,
) -> Path:
    parser = etree.XMLParser(remove_blank_text=False, recover=True, huge_tree=True)
    tree = etree.parse(str(svg_path), parser)
    root = tree.getroot()

    updated = 0
    not_found = 0
    skipped_blank = 0

    # sort for reproducibility and to process parent text before tspans consistently
    units_for_file = units_for_file.sort_values(
        ["element_path", "unit_type", "tspan_idx"],
        na_position="last"
    )

    for _, u in units_for_file.iterrows():
        new_text = u["translated_text"]
        if pd.isna(new_text):
            skipped_blank += 1
            continue

        el = find_by_element_path(root, u["element_path"])
        if el is None:
            not_found += 1
            continue

        if u["unit_type"] == "text":
            set_text_preserve_none(el, new_text)
            updated += 1

        elif u["unit_type"] == "tspan":
            idx = u.get("tspan_idx")
            if pd.isna(idx):
                not_found += 1
                continue

            tspans = el.findall(".//svg:tspan", namespaces=NS)
            idx = int(idx)

            if idx < 0 or idx >= len(tspans):
                not_found += 1
                continue

            set_text_preserve_none(tspans[idx], new_text)
            updated += 1

    out_path = output_dir / f"{svg_path.stem}_{target_language_slug}_{timestamp}{svg_path.suffix}"
    tree.write(str(out_path), encoding="utf-8", xml_declaration=True)

    print(
        f"{svg_path.name} -> {out_path.name} | "
        f"updated={updated} | not_found={not_found} | skipped_blank={skipped_blank}"
    )
    return out_path

In [30]:
df_apply["unit_type"].value_counts(dropna=False)

unit_type
text    341
Name: count, dtype: int64

In [31]:
df_apply[df_apply["translated_text"].notna()][
    ["source_file", "unit_type", "element_path", "tspan_idx", "source_text", "translated_text"]
].head(20)

,source_file,unit_type,element_path,tspan_idx,source_text,translated_text
0,tbe_01.svg,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,Adam 4174,亚当 4174
1,tbe_01.svg,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,Enoch 3552,以诺 3552
2,tbe_01.svg,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,Enosh 3939,以挪士 3939
3,tbe_01.svg,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,Jared 3714,雅列 3714
4,tbe_01.svg,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,Kenan 3849,该南 3849
5,tbe_01.svg,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,Lamech 3300,拉麦 3300
6,tbe_01.svg,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,Mahalalel 3779,玛勒列 3779
7,tbe_01.svg,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,Methuselah 3487,玛土撒拉 3487
8,tbe_01.svg,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,Noah 3118,挪亚 3118
9,tbe_01.svg,text,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,Seth 4044,塞特 4044


In [32]:
out_files = []

for source_file, units_for_file in df_apply.groupby("source_file", sort=True):
    svg_path = SVG_SOURCE_DIR / source_file
    out_file = apply_translations_to_svg(
        svg_path=svg_path,
        units_for_file=units_for_file,
        target_language_slug=target_language_slug,
        timestamp=timestamp,
        output_dir=SVG_OUTPUT_DIR,
    )
    out_files.append(out_file)

print("Wrote files:", len(out_files))
for p in out_files:
    print(" -", p.relative_to(PROJECT_ROOT))

tbe_01.svg -> tbe_01_simplified_chinese_20260516_0940.svg | updated=31 | not_found=0 | skipped_blank=0
tbe_02.svg -> tbe_02_simplified_chinese_20260516_0940.svg | updated=88 | not_found=0 | skipped_blank=0
tbe_03.svg -> tbe_03_simplified_chinese_20260516_0940.svg | updated=134 | not_found=0 | skipped_blank=0
tbe_04.svg -> tbe_04_simplified_chinese_20260516_0940.svg | updated=88 | not_found=0 | skipped_blank=0
Wrote files: 4
 - svg_output_files\tbe_01_simplified_chinese_20260516_0940.svg
 - svg_output_files\tbe_02_simplified_chinese_20260516_0940.svg
 - svg_output_files\tbe_03_simplified_chinese_20260516_0940.svg
 - svg_output_files\tbe_04_simplified_chinese_20260516_0940.svg


### Export translation table

In [33]:
from datetime import datetime
import pandas as pd
import re

def slugify_for_filename(s: str) -> str:
    s = (s or "").strip()
    return re.sub(r"[^\w]+", "_", s).strip("_").lower()

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
lang_slug = slugify_for_filename(target_language)

# Build export dataframe
df_export = (
    df_apply.loc[:, ["source_text", "translated_text", "group_stack"]]
    .copy()
)

print("Rows in export df:", len(df_export))
print("Missing translated_text:", df_export["translated_text"].isna().sum())

# Output paths
csv_path = SVG_OUTPUT_DIR / f"translations_table_{lang_slug}_{timestamp}.csv"
xlsx_path = SVG_OUTPUT_DIR / f"translations_table_{lang_slug}_{timestamp}.xlsx"
md_path = SVG_OUTPUT_DIR / f"translations_table_{lang_slug}_{timestamp}.md"

# Write files
df_export.to_csv(csv_path, index=False, encoding="utf-8-sig")
df_export.to_excel(xlsx_path, index=False)

md_text = df_export.to_markdown(index=False)
md_path.write_text(md_text, encoding="utf-8")

print("Wrote CSV :", csv_path.relative_to(PROJECT_ROOT))
print("Wrote XLSX:", xlsx_path.relative_to(PROJECT_ROOT))
print("Wrote MD  :", md_path.relative_to(PROJECT_ROOT))

df_export.head(10)

Rows in export df: 341
Missing translated_text: 0
Wrote CSV : svg_output_files\translations_table_simplified_chinese_20260516_0940.csv
Wrote XLSX: svg_output_files\translations_table_simplified_chinese_20260516_0940.xlsx
Wrote MD  : svg_output_files\translations_table_simplified_chinese_20260516_0940.md


,source_text,translated_text,group_stack
0,Adam 4174,亚当 4174,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...
1,Enoch 3552,以诺 3552,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...
2,Enosh 3939,以挪士 3939,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...
3,Jared 3714,雅列 3714,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...
4,Kenan 3849,该南 3849,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...
5,Lamech 3300,拉麦 3300,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...
6,Mahalalel 3779,玛勒列 3779,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...
7,Methuselah 3487,玛土撒拉 3487,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...
8,Noah 3118,挪亚 3118,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...
9,Seth 4044,塞特 4044,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...


## Collapse fragmented tspans

This section adds an SVG reconstruction path for text-level translation units. It replaces the full textual content of each matching `<text>` element, preserving the first `<tspan>` attributes when fragmented tspans are present and removing the extra original fragments from the output SVG.


In [34]:
from datetime import datetime
from pathlib import Path
from lxml import etree
import pandas as pd
import re

# Experimental reconstruction method for text-level translation units with fragmented tspans.
# Outputs are written to a separate directory so the existing reconstruction outputs are not overwritten.
COLLAPSED_TSPANS_OUTPUT_DIR = PROJECT_ROOT / "svg_output_files_collapsed_tspans"
assert COLLAPSED_TSPANS_OUTPUT_DIR != SVG_OUTPUT_DIR, "Collapsed-tspan output dir must be separate from SVG_OUTPUT_DIR."
COLLAPSED_TSPANS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLLAPSED_SVG_NS = globals().get("SVG_NS", "http://www.w3.org/2000/svg")
COLLAPSED_NS = globals().get("NS", {"svg": COLLAPSED_SVG_NS})
collapsed_target_language_slug = globals().get(
    "target_language_slug",
    re.sub(r"[^A-Za-z0-9]+", "_", target_language).strip("_").lower(),
)
collapsed_timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print("Collapsed-tspan output dir:", COLLAPSED_TSPANS_OUTPUT_DIR.relative_to(PROJECT_ROOT))


Collapsed-tspan output dir: svg_output_files_collapsed_tspans


In [35]:
def collapsed_localname(tag) -> str:
    """Return the local XML tag name without the namespace."""
    if "localname" in globals():
        return localname(tag)
    if not isinstance(tag, str):
        return ""
    return tag.split("}", 1)[1] if tag.startswith("{") else tag


def collapsed_find_by_element_path(root, path: str):
    """Resolve a stored SVG element_path using the existing notebook helper when available."""
    if "find_by_element_path" in globals():
        return find_by_element_path(root, path)

    if not path:
        return None

    parts = path.split("/")
    cur = root
    if not parts or not parts[0].startswith("svg"):
        return None

    for part in parts[1:]:
        m = re.fullmatch(r"([A-Za-z0-9_:-]+)(?:#(.+)|\[(\d+)\])?", part)
        if not m:
            return None

        tag, el_id, idx = m.groups()
        candidates = [
            child for child in cur
            if isinstance(getattr(child, "tag", None), str) and collapsed_localname(child.tag) == tag
        ]

        if el_id is not None:
            cur = next((child for child in candidates if child.get("id") == el_id), None)
            if cur is None:
                return None
        elif idx is not None:
            idx0 = int(idx) - 1
            if idx0 < 0 or idx0 >= len(candidates):
                return None
            cur = candidates[idx0]
        else:
            if not candidates:
                return None
            cur = candidates[0]

    return cur


def collapsed_translated_text_column(df: pd.DataFrame) -> str:
    """Infer the translated-text column without changing upstream data structures."""
    candidates = [
        "translated_text",
        "final_text",
        "final_translated_text",
        "target_text",
        "translation",
    ]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"Could not find a translated-text column. Available columns: {list(df.columns)}")


def collapsed_build_text_lookup(units_for_file: pd.DataFrame) -> dict[str, str]:
    """Build element_path -> translated text for text-level units in one source SVG file."""
    text_col = collapsed_translated_text_column(units_for_file)
    work = units_for_file.copy()

    if "unit_type" in work.columns:
        work = work[work["unit_type"].fillna("").astype(str).eq("text")].copy()

    work = work[
        work["element_path"].notna()
        & work[text_col].notna()
        & work[text_col].astype(str).str.strip().ne("")
    ].copy()

    lookup = {}
    for _, row in work.sort_values("element_path").iterrows():
        lookup[str(row["element_path"])] = str(row[text_col])
    return lookup


In [36]:
def collapsed_apply_text_level_translations_to_svg(
    svg_path: Path,
    units_for_file: pd.DataFrame,
    target_language_slug: str,
    timestamp: str,
    output_dir: Path,
) -> tuple[Path, dict]:
    """
    Experimental reconstruction for text-level SVG units.

    If a translated <text> element contains direct <tspan> children, the full translated
    string is placed in the first tspan, all other direct tspans are removed, and the
    first tspan's attributes are preserved. The translated text is never split back
    across fragmented tspans.
    """
    parser = etree.XMLParser(remove_blank_text=False, recover=True, huge_tree=True)
    tree = etree.parse(str(svg_path), parser)
    root = tree.getroot()

    text_lookup = collapsed_build_text_lookup(units_for_file)

    applied = 0
    collapsed_text_elements = 0
    removed_extra_tspans = 0
    unmatched = 0

    for element_path, translated_text in text_lookup.items():
        text_el = collapsed_find_by_element_path(root, element_path)
        if text_el is None or collapsed_localname(text_el.tag) != "text":
            unmatched += 1
            continue

        direct_tspans = [
            child for child in text_el
            if isinstance(getattr(child, "tag", None), str) and collapsed_localname(child.tag) == "tspan"
        ]

        if direct_tspans:
            first_tspan = direct_tspans[0]
            first_tspan.text = translated_text
            first_tspan.tail = None
            text_el.text = None

            for extra_tspan in direct_tspans[1:]:
                text_el.remove(extra_tspan)
                removed_extra_tspans += 1

            collapsed_text_elements += 1
        else:
            text_el.text = translated_text

        applied += 1

    out_path = output_dir / f"{svg_path.stem}_{target_language_slug}_{timestamp}_collapsed_tspans{svg_path.suffix}"
    tree.write(str(out_path), encoding="utf-8", xml_declaration=True)

    stats = {
        "source_svg_filename": svg_path.name,
        "output_svg_filename": out_path.name,
        "translated_text_elements_applied": applied,
        "text_elements_with_tspans_collapsed": collapsed_text_elements,
        "extra_tspans_removed": removed_extra_tspans,
        "translated_units_not_matched_to_text_element": unmatched,
    }
    return out_path, stats


def collapsed_print_reconstruction_stats(stats: dict) -> None:
    """Print notebook-friendly diagnostics for one collapsed-tspan reconstruction run."""
    print("source SVG filename:", stats["source_svg_filename"])
    print("output SVG filename:", stats["output_svg_filename"])
    print("number of translated text elements applied:", stats["translated_text_elements_applied"])
    print("number of text elements where tspans were collapsed:", stats["text_elements_with_tspans_collapsed"])
    print("number of extra tspans removed:", stats["extra_tspans_removed"])
    print("number of translated units not matched to a text element:", stats["translated_units_not_matched_to_text_element"])


In [37]:
# Smoke test: reconstruct only the first source SVG with translation results available.
collapsed_df_apply = df_apply.copy()
collapsed_text_col = collapsed_translated_text_column(collapsed_df_apply)

collapsed_available = collapsed_df_apply[
    collapsed_df_apply["source_file"].notna()
    & collapsed_df_apply["element_path"].notna()
    & collapsed_df_apply[collapsed_text_col].notna()
    & collapsed_df_apply[collapsed_text_col].astype(str).str.strip().ne("")
].copy()

if "unit_type" in collapsed_available.columns:
    collapsed_available = collapsed_available[
        collapsed_available["unit_type"].fillna("").astype(str).eq("text")
    ].copy()

collapsed_source_file = next(
    source_file for source_file in sorted(collapsed_available["source_file"].unique())
    if (SVG_SOURCE_DIR / source_file).exists()
)

collapsed_smoke_units = collapsed_df_apply[collapsed_df_apply["source_file"].eq(collapsed_source_file)].copy()
collapsed_smoke_svg_path = SVG_SOURCE_DIR / collapsed_source_file

collapsed_smoke_out_path, collapsed_smoke_stats = collapsed_apply_text_level_translations_to_svg(
    svg_path=collapsed_smoke_svg_path,
    units_for_file=collapsed_smoke_units,
    target_language_slug=collapsed_target_language_slug,
    timestamp=collapsed_timestamp,
    output_dir=COLLAPSED_TSPANS_OUTPUT_DIR,
)

collapsed_print_reconstruction_stats(collapsed_smoke_stats)
print("wrote smoke-test SVG:", collapsed_smoke_out_path.relative_to(PROJECT_ROOT))


source SVG filename: tbe_01.svg
output SVG filename: tbe_01_simplified_chinese_20260516_0953_collapsed_tspans.svg
number of translated text elements applied: 31
number of text elements where tspans were collapsed: 1
number of extra tspans removed: 1
number of translated units not matched to a text element: 0
wrote smoke-test SVG: svg_output_files_collapsed_tspans\tbe_01_simplified_chinese_20260516_0953_collapsed_tspans.svg


In [38]:
# Inspect the smoke-test SVG for source-language fragments that should have been removed
# when extra fragmented tspans were deleted.
collapsed_smoke_fragments = ["dam 4174", "ndus", "Valley"]
collapsed_smoke_svg_text = collapsed_smoke_out_path.read_text(encoding="utf-8", errors="replace")
collapsed_remaining_fragments = [frag for frag in collapsed_smoke_fragments if frag in collapsed_smoke_svg_text]

if collapsed_remaining_fragments:
    print("WARNING: source fragments remain in collapsed smoke-test output:", collapsed_remaining_fragments)
else:
    print("Smoke-test fragment check passed: no configured source fragments remain.")


Smoke-test fragment check passed: no configured source fragments remain.


In [39]:
# Optional full run: reconstruct every source SVG into the separate collapsed-tspan output folder.
# Run this cell after the smoke test looks correct.
collapsed_out_files = []
collapsed_all_stats = []

for source_file, units_for_file in df_apply.groupby("source_file", sort=True):
    svg_path = SVG_SOURCE_DIR / source_file
    if not svg_path.exists():
        print("Skipping missing source SVG:", source_file)
        continue

    out_path, stats = collapsed_apply_text_level_translations_to_svg(
        svg_path=svg_path,
        units_for_file=units_for_file,
        target_language_slug=collapsed_target_language_slug,
        timestamp=collapsed_timestamp,
        output_dir=COLLAPSED_TSPANS_OUTPUT_DIR,
    )
    collapsed_out_files.append(out_path)
    collapsed_all_stats.append(stats)
    print(f"{source_file} -> {out_path.name}")

print("Wrote collapsed-tspan files:", len(collapsed_out_files))
print("Total translated text elements applied:", sum(s["translated_text_elements_applied"] for s in collapsed_all_stats))
print("Total text elements where tspans were collapsed:", sum(s["text_elements_with_tspans_collapsed"] for s in collapsed_all_stats))
print("Total extra tspans removed:", sum(s["extra_tspans_removed"] for s in collapsed_all_stats))
print("Total translated units not matched to text element:", sum(s["translated_units_not_matched_to_text_element"] for s in collapsed_all_stats))


tbe_01.svg -> tbe_01_simplified_chinese_20260516_0953_collapsed_tspans.svg
tbe_02.svg -> tbe_02_simplified_chinese_20260516_0953_collapsed_tspans.svg
tbe_03.svg -> tbe_03_simplified_chinese_20260516_0953_collapsed_tspans.svg
tbe_04.svg -> tbe_04_simplified_chinese_20260516_0953_collapsed_tspans.svg
Wrote collapsed-tspan files: 4
Total translated text elements applied: 341
Total text elements where tspans were collapsed: 31
Total extra tspans removed: 68
Total translated units not matched to text element: 0
